### RESULTS
Cargaremos el rendimiento de los modelos entrenados y generaremos tablas y figuras comparativas entre los distintos modelos.

In [ ]:
# Importaciones
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import numpy as np
import os
import glob
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import LabelBinarizer
from itertools import cycle

### Configuración

In [ ]:
# Configuración estética de las gráficas
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Carpeta de salida para las gráficas
OUTPUT_FOLDER = "graficas_resultados"
# Crear la carpeta si no existe
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Carpeta de salida configurada: {os.path.abspath(OUTPUT_FOLDER)}")

### Carga de los Resultados

In [ ]:
try:
    df_metrics = pd.read_csv("evaluacion_metricas.csv")
    with open("datos_para_roc.pkl", "rb") as f:
        datos_modelos = pickle.load(f)
    print("¡Datos cargados correctamente!")
except FileNotFoundError as e:
    print(f"Error: No se encuentra el archivo {e.filename}. Asegúrate de haber ejecutado eval.py primero.")

# Mostrar tabla numérica
print("\n--- TABLA COMPARATIVA DE MÉTRICAS ---")
display(df_metrics)

### Comparativa de Modelos (Barras)

In [ ]:
def plot_model_comparison(df):
    # Seleccionamos las métricas más importantes para graficar
    metrics_to_plot = ['Accuracy', 'F1-Score', 'AUC', 'Recall (Sensibilidad)']
    
    # Transformamos el DF para que sea fácil de graficar con Seaborn (formato "largo")
    df_melted = df.melt(id_vars="Modelo", value_vars=metrics_to_plot, var_name="Métrica", value_name="Valor")
    
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(data=df_melted, x="Métrica", y="Valor", hue="Modelo", palette="viridis")
    
    plt.title("Comparación de Rendimiento entre Modelos YOLO (N, S, M)")
    plt.ylim(0, 1.1) # Escala de 0 a 1.1 para ver bien las barras
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Añadir el valor numérico encima de cada barra
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=3, fontsize=10)
        
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_FOLDER, "comparativa_modelos.png")
    plt.savefig(save_path)
    print(f"Gráfica guardada: {save_path}")
    plt.show()

print("\nGenerando gráfica comparativa...")
plot_model_comparison(df_metrics)

### Matrices de Confusión y Curvas ROC

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, model_name):
    """Genera y visualiza la matriz de confusión normalizada"""
    cm = confusion_matrix(y_true, y_pred)
    # Normalizar por filas (para ver porcentajes de acierto por clase real)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=list(classes.values()), 
                yticklabels=list(classes.values()))
    
    plt.title(f'Matriz de Confusión Normalizada - Modelo {model_name}')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Etiqueta Predicha')
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_FOLDER, f"conf_matrix_{model_name}.png")
    plt.savefig(save_path)
    print(f"Gráfica guardada: {save_path}")
    plt.show()


def plot_multiclass_roc(y_true, y_probs, classes, model_name):
    """Genera curvas ROC para cada clase (Estrategia One-vs-Rest)"""
    
    # Binarizar etiquetas reales
    lb = LabelBinarizer()
    lb.fit(y_true)
    y_true_bin = lb.transform(y_true)
    n_classes = y_true_bin.shape[1]
    
    # Obtener nombres de clases en orden
    class_names = [classes[i] for i in range(n_classes)]
    
    # Calcular ROC y AUC para cada clase
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        
    # Graficar
    plt.figure(figsize=(10, 8))
    colors = cycle(['blue', 'red', 'green', 'orange', 'purple', 'cyan', 'magenta', 'yellow'])
    
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')
    
    plt.plot([0, 1], [0, 1], 'k--', lw=2) # Línea diagonal de azar
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
    plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
    plt.title(f'Curvas ROC Multiclase - Modelo {model_name}')
    plt.legend(loc="lower right")
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_FOLDER, f"roc_curve_{model_name}.png")
    plt.savefig(save_path)
    print(f"Gráfica guardada: {save_path}")
    plt.show()


# Bucle principal para generar gráficas por cada modelo
for model_name, data in datos_modelos.items():
    print(f"\nGenerando visualizaciones para: {model_name}...")
    
    y_true = data['y_true']
    # Para la matriz de confusión necesitamos las predicciones "duras" (clase 0, 1, 2...)
    # Las calculamos sacando el índice de la probabilidad máxima
    y_pred = np.argmax(data['y_probs'], axis=1)
    y_probs = data['y_probs']
    classes = data['class_names']
    
    # 1. Matriz de Confusión
    plot_confusion_matrix(y_true, y_pred, classes, model_name)
    
    # 2. Curvas ROC
    try:
        plot_multiclass_roc(y_true, y_probs, classes, model_name)
    except Exception as e:
        print(f"No se pudo generar ROC para {model_name}: {e}")

### Análisis del Entrenamiento (Curvas de Loss)

In [ ]:
def plot_training_curves(runs_dir="runs/classify"):
    print("\nGenerando gráficas de entrenamiento (Loss)...")
    
    # Buscar todos los archivos results.csv en las carpetas de entrenamiento
    # La estructura típica es: runs/classify/train_nombre/results.csv
    csv_files = glob.glob(f"{runs_dir}/*/results.csv")
    
    if not csv_files:
        print("No se encontraron archivos results.csv de entrenamiento.")
        return

    # Preparamos la figura
    plt.figure(figsize=(14, 6))
    
    # Colores para distinguir modelos
    colors = cycle(['blue', 'orange', 'green', 'red', 'purple'])
    
    for csv_file, color in zip(csv_files, colors):
        try:
            # Extraer nombre del modelo de la ruta
            # Ej: runs/classify/train_yolov8n-cls/results.csv -> yolov8n-cls
            model_name = os.path.dirname(csv_file).split(os.sep)[-1].replace("train_", "").replace(".pt", "")
            
            # Cargar datos
            df = pd.read_csv(csv_file)
            
            # Limpiar nombres de columnas (YOLO pone espacios extra: " train/loss")
            df.columns = df.columns.str.strip()
            
            # Verificar que existan las columnas de loss
            if 'train/loss' in df.columns and 'val/loss' in df.columns:
                epochs = df['epoch'] if 'epoch' in df.columns else df.index
                
                # Pintar línea continua para Train
                plt.plot(epochs, df['train/loss'], label=f'{model_name} (Train)', 
                         linestyle='-', color=color, linewidth=2)
                
                # Pintar línea discontinua para Val
                plt.plot(epochs, df['val/loss'], label=f'{model_name} (Val)', 
                         linestyle='--', color=color, linewidth=2)
            else:
                print(f"Advertencia: El archivo {csv_file} no tiene columnas de loss estándar.")
                
        except Exception as e:
            print(f"Error procesando {csv_file}: {e}")

    plt.title("Evolución de la Pérdida (Loss) durante el Entrenamiento")
    plt.xlabel("Épocas")
    plt.ylabel("Pérdida (Loss)")
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    
    # Guardar en la carpeta de gráficas
    save_path = os.path.join(OUTPUT_FOLDER, "curvas_entrenamiento_loss.png")
    plt.savefig(save_path)
    print(f"Gráfica de Loss guardada: {save_path}")
    plt.show()

# Ejecutar la función
plot_training_curves()